# 🛒 Retail Sales Forecasting System
> **End-to-End Machine Learning Pipeline**  
> Predict future retail sales using historical data with Python, Scikit-learn & Matplotlib.

---

### 📌 What this notebook covers:
| Step | Description |
|------|-------------|
| 1 | Install dependencies |
| 2 | Generate synthetic retail sales dataset |
| 3 | Exploratory Data Analysis (EDA) |
| 4 | Data Preprocessing |
| 5 | Feature Engineering (Lag + Rolling) |
| 6 | Model Training (Random Forest) |
| 7 | Model Evaluation (MAE, RMSE, R²) |
| 8 | Feature Importance |
| 9 | Sales Prediction |
| 10 | 7-Day Forecast Chart |

---
## ⚙️ Step 1 — Install Dependencies

In [ ]:
# All core libraries are pre-installed in Colab.
# This cell confirms versions and installs any missing ones.
!pip install -q scikit-learn pandas numpy matplotlib seaborn joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import joblib
import os
import warnings

from datetime import datetime, timedelta
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

print('✅ All libraries loaded successfully!')
print(f'  pandas      : {pd.__version__}')
print(f'  numpy       : {np.__version__}')
print(f'  scikit-learn: {__import__("sklearn").__version__}')
print(f'  seaborn     : {sns.__version__}')

---
## 📦 Step 2 — Generate Synthetic Retail Sales Dataset

We generate **3 years** of daily sales data (2022–2024) across **3 stores** and **4 products**.

| Feature | Detail |
|---------|--------|
| Date range | 2022-01-01 → 2024-12-31 |
| Stores | 3 (IDs: 1, 2, 3) |
| Products | 4 (IDs: 101, 102, 103, 104) |
| Total rows | ~13,000 |
| Patterns | Seasonal, weekend boost, promo effect, holidays |

In [ ]:
np.random.seed(42)

start_date = datetime(2022, 1, 1)
end_date   = datetime(2024, 12, 31)
dates      = pd.date_range(start=start_date, end=end_date, freq='D')

store_ids   = [1, 2, 3]
product_ids = [101, 102, 103, 104]

rows = []
for store in store_ids:
    for product in product_ids:
        base_sales = np.random.randint(100, 300)
        for date in dates:
            promotion    = np.random.choice([0, 1], p=[0.8, 0.2])
            holiday      = 1 if date.weekday() == 6 or (date.month == 12 and date.day == 25) else 0
            seasonal     = 20 * np.sin(2 * np.pi * date.dayofyear / 365)
            weekend_boost = 30 if date.weekday() >= 5 else 0
            promo_boost  = 50 * promotion
            noise        = np.random.normal(0, 15)
            sales        = max(0, int(base_sales + seasonal + weekend_boost + promo_boost + noise))
            rows.append([date.strftime('%Y-%m-%d'), store, product, sales, promotion, holiday])

df_raw = pd.DataFrame(rows, columns=['date', 'store_id', 'product_id', 'sales', 'promotion', 'holiday'])

print(f'✅ Dataset generated: {len(df_raw):,} rows × {df_raw.shape[1]} columns')
print(f'   Date range : {df_raw["date"].min()}  →  {df_raw["date"].max()}')
print(f'   Stores     : {df_raw["store_id"].unique().tolist()}')
print(f'   Products   : {df_raw["product_id"].unique().tolist()}')
print()
df_raw.head(10)

---
## 📊 Step 3 — Exploratory Data Analysis (EDA)

In [ ]:
# ── Basic Statistics ──────────────────────────────────────────────────────────
print('=' * 55)
print('  Dataset Info')
print('=' * 55)
df_raw.info()
print()
print('=' * 55)
print('  Descriptive Statistics')
print('=' * 55)
df_raw.describe().round(2)

In [ ]:
# ── Missing Values ────────────────────────────────────────────────────────────
print('Missing values per column:')
print(df_raw.isnull().sum())
print(f'\nTotal missing: {df_raw.isnull().sum().sum()}')

In [ ]:
# ── Sales Distribution ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogram
axes[0].hist(df_raw['sales'], bins=40, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title('Sales Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Sales (units)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df_raw['sales'].mean(), color='tomato', linestyle='--', label=f'Mean: {df_raw["sales"].mean():.0f}')
axes[0].legend()

# Boxplot by store
df_raw.boxplot(column='sales', by='store_id', ax=axes[1], grid=False,
               boxprops=dict(color='steelblue'),
               medianprops=dict(color='tomato', linewidth=2))
axes[1].set_title('Sales by Store', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Store ID')
axes[1].set_ylabel('Sales (units)')
plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
# ── Monthly Sales Trend ───────────────────────────────────────────────────────
df_eda = df_raw.copy()
df_eda['date'] = pd.to_datetime(df_eda['date'])
df_eda['month_year'] = df_eda['date'].dt.to_period('M')

monthly = df_eda.groupby('month_year')['sales'].mean().reset_index()
monthly['month_year'] = monthly['month_year'].astype(str)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(monthly['month_year'], monthly['sales'], color='steelblue', linewidth=2, marker='o', markersize=4)
ax.fill_between(monthly['month_year'], monthly['sales'], alpha=0.15, color='steelblue')
ax.set_title('Average Monthly Sales Trend (All Stores & Products)', fontsize=13, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Avg Sales (units)')
plt.xticks(rotation=45, ha='right', fontsize=8)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# ── Sales by Day of Week ──────────────────────────────────────────────────────
df_eda['day_of_week'] = df_eda['date'].dt.day_name()
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_avg = df_eda.groupby('day_of_week')['sales'].mean().reindex(dow_order)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

colors = ['#a5b4fc' if d not in ['Saturday', 'Sunday'] else '#667eea' for d in dow_order]
axes[0].bar(dow_order, dow_avg.values, color=colors, edgecolor='white')
axes[0].set_title('Avg Sales by Day of Week', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Day')
axes[0].set_ylabel('Avg Sales')
axes[0].spines[['top', 'right']].set_visible(False)
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30, ha='right')

# Promotion effect
promo_avg = df_eda.groupby('promotion')['sales'].mean()
axes[1].bar(['No Promotion', 'Promotion'], promo_avg.values,
            color=['#94a3b8', '#667eea'], edgecolor='white', width=0.4)
axes[1].set_title('Sales: Promotion vs No Promotion', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Avg Sales')
for i, v in enumerate(promo_avg.values):
    axes[1].text(i, v + 1, f'{v:.0f}', ha='center', fontweight='bold')
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# ── Sales by Product ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

product_avg = df_raw.groupby('product_id')['sales'].mean()
axes[0].bar([f'Product {p}' for p in product_avg.index], product_avg.values,
            color='#667eea', edgecolor='white')
axes[0].set_title('Avg Sales by Product', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Avg Sales')
axes[0].spines[['top', 'right']].set_visible(False)

# Correlation heatmap
corr_cols = ['sales', 'promotion', 'holiday']
df_eda['month']   = df_eda['date'].dt.month
df_eda['weekday'] = df_eda['date'].dt.dayofweek
corr_data = df_eda[corr_cols + ['month', 'weekday']].corr()
sns.heatmap(corr_data, annot=True, fmt='.2f', cmap='coolwarm',
            ax=axes[1], linewidths=0.5, square=True)
axes[1].set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── Store 1 / Product 101 — Full Time Series ──────────────────────────────────
ts = df_eda[(df_eda['store_id'] == 1) & (df_eda['product_id'] == 101)].copy()
ts = ts.sort_values('date')

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(ts['date'], ts['sales'], color='steelblue', linewidth=0.8, alpha=0.8)
ax.fill_between(ts['date'], ts['sales'], alpha=0.1, color='steelblue')

# Highlight promotions
promo_mask = ts['promotion'] == 1
ax.scatter(ts.loc[promo_mask, 'date'], ts.loc[promo_mask, 'sales'],
           color='tomato', s=10, zorder=5, label='Promotion days')

ax.set_title('Daily Sales — Store 1, Product 101 (2022–2024)', fontsize=13, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Sales (units)')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

---
## 🔧 Step 4 — Data Preprocessing

Steps:
1. Convert `date` column to datetime
2. Extract temporal features (year, month, day, day_of_week, is_weekend, quarter, day_of_year)
3. Handle missing values (median for numeric, mode for categorical)
4. Label-encode any remaining categorical columns

In [ ]:
def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    """Full preprocessing pipeline."""
    df = df.copy()

    # 1. Convert date
    df['date'] = pd.to_datetime(df['date'])
    print('[INFO] Converted date column to datetime.')

    # 2. Extract date features
    df['year']       = df['date'].dt.year
    df['month']      = df['date'].dt.month
    df['day']        = df['date'].dt.day
    df['day_of_week']= df['date'].dt.dayofweek
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    df['quarter']    = df['date'].dt.quarter
    df['day_of_year']= df['date'].dt.dayofyear
    print('[INFO] Extracted: year, month, day, day_of_week, is_weekend, quarter, day_of_year.')

    # 3. Handle missing values
    before = df.isnull().sum().sum()
    for col in df.select_dtypes(include=[np.number]).columns:
        df[col] = df[col].fillna(df[col].median())
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].fillna(df[col].mode()[0])
    after = df.isnull().sum().sum()
    print(f'[INFO] Missing values: {before} → {after}')

    # 4. Encode categoricals (skip date)
    cat_cols = [c for c in df.select_dtypes(include=['object']).columns if c != 'date']
    for col in cat_cols:
        df[col] = df[col].astype('category').cat.codes
        print(f'[INFO] Label-encoded: {col}')

    print(f'[INFO] Preprocessing complete. Shape: {df.shape}')
    return df


df_processed = preprocess(df_raw)
print()
df_processed.head()

---
## ⚙️ Step 5 — Feature Engineering

We create **time-series features** that help the model understand past patterns:

| Feature | Description |
|---------|-------------|
| `sales_lag_1` | Sales 1 day ago (short-term trend) |
| `sales_lag_7` | Sales 7 days ago (weekly seasonality) |
| `sales_lag_14` | Sales 14 days ago (bi-weekly pattern) |
| `rolling_mean_7` | 7-day rolling average (smoothed trend) |
| `rolling_std_7` | 7-day rolling std deviation (volatility) |
| `rolling_mean_30` | 30-day rolling average (monthly trend) |

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add lag and rolling window features."""
    df = df.sort_values(by=['store_id', 'product_id', 'date']).reset_index(drop=True)
    print('[INFO] Data sorted by store_id, product_id, date.')

    group = df.groupby(['store_id', 'product_id'])['sales']

    # Lag features
    df['sales_lag_1']  = group.shift(1)
    df['sales_lag_7']  = group.shift(7)
    df['sales_lag_14'] = group.shift(14)
    print('[INFO] Added lag features: sales_lag_1, sales_lag_7, sales_lag_14.')

    # Rolling features
    df['rolling_mean_7']  = group.transform(lambda x: x.shift(1).rolling(7,  min_periods=1).mean())
    df['rolling_std_7']   = group.transform(lambda x: x.shift(1).rolling(7,  min_periods=1).std())
    df['rolling_mean_30'] = group.transform(lambda x: x.shift(1).rolling(30, min_periods=1).mean())
    print('[INFO] Added rolling features: rolling_mean_7, rolling_std_7, rolling_mean_30.')

    # Drop rows with NaN (introduced by lags)
    before = len(df)
    df = df.dropna().reset_index(drop=True)
    print(f'[INFO] Dropped {before - len(df)} NaN rows. Remaining: {len(df):,}')
    print(f'[INFO] Feature engineering complete. Shape: {df.shape}')
    return df


df_featured = engineer_features(df_processed)
print()
df_featured[['date', 'store_id', 'product_id', 'sales',
             'sales_lag_1', 'sales_lag_7', 'rolling_mean_7', 'rolling_mean_30']].head(10)

In [ ]:
# Visualise lag feature correlation with sales
lag_cols = ['sales', 'sales_lag_1', 'sales_lag_7', 'sales_lag_14',
            'rolling_mean_7', 'rolling_std_7', 'rolling_mean_30']
corr_lag = df_featured[lag_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_lag, annot=True, fmt='.2f', cmap='YlOrRd',
            ax=ax, linewidths=0.5, square=True)
ax.set_title('Lag & Rolling Feature Correlation with Sales', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🤖 Step 6 — Model Training

**Algorithm:** Random Forest Regressor  
**Split:** Time-based (last 20% of dates = test set)  
**Features:** 17 total (temporal + lag + rolling + store/product/promo/holiday)

In [ ]:
FEATURE_COLS = [
    'store_id', 'product_id',
    'year', 'month', 'day', 'day_of_week',
    'is_weekend', 'quarter', 'day_of_year',
    'promotion', 'holiday',
    'sales_lag_1', 'sales_lag_7', 'sales_lag_14',
    'rolling_mean_7', 'rolling_std_7', 'rolling_mean_30'
]
TARGET_COL = 'sales'

# ── Time-based train/test split ───────────────────────────────────────────────
df_sorted  = df_featured.sort_values('date')
split_idx  = int(len(df_sorted) * 0.80)
train_df   = df_sorted.iloc[:split_idx]
test_df    = df_sorted.iloc[split_idx:]

X_train = train_df[FEATURE_COLS]
y_train = train_df[TARGET_COL]
X_test  = test_df[FEATURE_COLS]
y_test  = test_df[TARGET_COL]

print('=' * 55)
print('  Train / Test Split')
print('=' * 55)
print(f'  Train: {len(X_train):>6,} samples  |  {train_df["date"].min().date()} → {train_df["date"].max().date()}')
print(f'  Test : {len(X_test):>6,} samples  |  {test_df["date"].min().date()} → {test_df["date"].max().date()}')
print(f'  Features: {len(FEATURE_COLS)}')

In [ ]:
print('=' * 55)
print('  Training RandomForestRegressor...')
print('=' * 55)

model = RandomForestRegressor(
    n_estimators=150,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)
print('  ✅ Model training complete!')
print(f'  Trees     : {model.n_estimators}')
print(f'  Max depth : {model.max_depth}')

---
## 📈 Step 7 — Model Evaluation

In [ ]:
y_pred = model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print('=' * 55)
print('  Model Evaluation — Test Set')
print('=' * 55)
print(f'  ┌─────────────────────────────────────┐')
print(f'  │  Mean Absolute Error  (MAE) : {mae:>7.2f} │')
print(f'  │  Root Mean Sq Error  (RMSE) : {rmse:>7.2f} │')
print(f'  │  R² Score                   : {r2:>7.4f} │')
print(f'  └─────────────────────────────────────┘')

In [ ]:
# ── Actual vs Predicted Plot ──────────────────────────────────────────────────
sample = test_df[test_df['store_id'] == 1].head(120).copy()
preds_sample = y_pred[:len(sample)]

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Time series comparison
axes[0].plot(sample['date'].values, sample['sales'].values,
             label='Actual', color='steelblue', linewidth=1.5)
axes[0].plot(sample['date'].values, preds_sample,
             label='Predicted', color='tomato', linestyle='--', linewidth=1.5)
axes[0].fill_between(sample['date'].values,
                     sample['sales'].values, preds_sample,
                     alpha=0.1, color='tomato')
axes[0].set_title('Actual vs Predicted Sales — Store 1 (Test Set)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Sales (units)')
axes[0].legend()
axes[0].spines[['top', 'right']].set_visible(False)

# Residuals
residuals = y_test.values - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.3, color='steelblue', s=8)
axes[1].axhline(0, color='tomato', linestyle='--', linewidth=1.5)
axes[1].set_title('Residual Plot (Predicted vs Error)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Predicted Sales')
axes[1].set_ylabel('Residual')
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# ── Scatter: Actual vs Predicted ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test, y_pred, alpha=0.3, color='steelblue', s=8)

# Perfect prediction line
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')

ax.set_title(f'Actual vs Predicted  (R² = {r2:.4f})', fontsize=13, fontweight='bold')
ax.set_xlabel('Actual Sales')
ax.set_ylabel('Predicted Sales')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

---
## 🏆 Step 8 — Feature Importance

In [ ]:
importances = pd.Series(model.feature_importances_, index=FEATURE_COLS)
importances = importances.sort_values(ascending=False)

print('  Top Feature Importances:')
print('  ' + '-' * 45)
for feat, imp in importances.items():
    bar = '█' * int(imp * 60)
    print(f'  {feat:<22} {bar}  ({imp:.4f})')

# Plot
fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#667eea' if i < 5 else '#a5b4fc' for i in range(len(importances))]
bars = ax.barh(importances.index[::-1], importances.values[::-1],
               color=colors[::-1], edgecolor='white')
ax.set_title('Feature Importance — Random Forest', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.spines[['top', 'right']].set_visible(False)
for bar, val in zip(bars, importances.values[::-1]):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=8)
plt.tight_layout()
plt.show()

---
## 🔮 Step 9 — Sales Prediction

Use the trained model to predict sales for any store/product/date combination.

In [ ]:
def predict_sales(store_id: int, product_id: int, date: str,
                  promotion: int, holiday: int,
                  sales_lag_1=None, sales_lag_7=None, sales_lag_14=None,
                  rolling_mean_7=None, rolling_std_7=None, rolling_mean_30=None) -> int:
    """
    Predict retail sales for given inputs.

    Parameters
    ----------
    store_id    : int  — Store ID (1, 2, or 3)
    product_id  : int  — Product ID (101, 102, 103, or 104)
    date        : str  — Forecast date 'YYYY-MM-DD'
    promotion   : int  — 1 if promo active, else 0
    holiday     : int  — 1 if public holiday, else 0
    sales_lag_* : float (optional) — historical sales for better accuracy

    Returns
    -------
    int — predicted sales units
    """
    dt           = pd.to_datetime(date)
    default_sales = 200  # fallback if lag values not provided

    row = {
        'store_id'       : store_id,
        'product_id'     : product_id,
        'year'           : dt.year,
        'month'          : dt.month,
        'day'            : dt.day,
        'day_of_week'    : dt.dayofweek,
        'is_weekend'     : int(dt.dayofweek >= 5),
        'quarter'        : dt.quarter,
        'day_of_year'    : dt.dayofyear,
        'promotion'      : promotion,
        'holiday'        : holiday,
        'sales_lag_1'    : sales_lag_1    if sales_lag_1    is not None else default_sales,
        'sales_lag_7'    : sales_lag_7    if sales_lag_7    is not None else default_sales,
        'sales_lag_14'   : sales_lag_14   if sales_lag_14   is not None else default_sales,
        'rolling_mean_7' : rolling_mean_7 if rolling_mean_7 is not None else default_sales,
        'rolling_std_7'  : rolling_std_7  if rolling_std_7  is not None else 15.0,
        'rolling_mean_30': rolling_mean_30 if rolling_mean_30 is not None else default_sales,
    }

    X = pd.DataFrame([row])[FEATURE_COLS]
    return max(0, round(int(model.predict(X)[0])))


print('predict_sales() function ready.')

In [ ]:
# ── Single Prediction Examples ────────────────────────────────────────────────
test_cases = [
    {'store_id': 1, 'product_id': 101, 'date': '2025-06-15', 'promotion': 1, 'holiday': 0},
    {'store_id': 2, 'product_id': 102, 'date': '2025-12-25', 'promotion': 0, 'holiday': 1},
    {'store_id': 3, 'product_id': 103, 'date': '2025-03-22', 'promotion': 0, 'holiday': 0},
    {'store_id': 1, 'product_id': 104, 'date': '2025-11-28', 'promotion': 1, 'holiday': 1},
]

print('=' * 65)
print(f'  {"Store":<8} {"Product":<12} {"Date":<14} {"Promo":<8} {"Holiday":<10} {"Predicted"}')
print('  ' + '-' * 62)
for tc in test_cases:
    pred = predict_sales(**tc)
    day  = pd.to_datetime(tc['date']).strftime('%a')
    print(f'  Store {tc["store_id"]}   Product {tc["product_id"]}   '
          f'{tc["date"]} ({day})   {"Yes" if tc["promotion"] else "No":<8} '
          f'{"Yes" if tc["holiday"] else "No":<10} {pred:>5} units')
print('=' * 65)

---
## 📊 Step 10 — 7-Day Sales Forecast Chart

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
FORECAST_STORE   = 1       # Store to forecast (1, 2, or 3)
FORECAST_PRODUCT = 101     # Product to forecast (101, 102, 103, 104)
FORECAST_START   = '2025-06-16'  # Start date for 7-day forecast
PROMO_ACTIVE     = 0       # 1 = promotion active, 0 = no promotion

# ── Generate 7-day forecast ───────────────────────────────────────────────────
start = pd.to_datetime(FORECAST_START)
forecast_dates = [start + timedelta(days=i) for i in range(7)]

forecasts = []
for fd in forecast_dates:
    is_holiday = 1 if fd.weekday() == 6 else 0
    pred = predict_sales(
        store_id=FORECAST_STORE,
        product_id=FORECAST_PRODUCT,
        date=str(fd.date()),
        promotion=PROMO_ACTIVE,
        holiday=is_holiday
    )
    forecasts.append({'date': fd, 'day': fd.strftime('%b %d\n%a'), 'sales': pred, 'holiday': is_holiday})

forecast_df = pd.DataFrame(forecasts)

print(f'7-Day Forecast — Store {FORECAST_STORE}, Product {FORECAST_PRODUCT}')
print(f'Start date: {FORECAST_START}   Promotion: {"Yes" if PROMO_ACTIVE else "No"}')
print('-' * 40)
for _, row in forecast_df.iterrows():
    flag = ' 🎉 Holiday' if row['holiday'] else ''
    print(f'  {row["date"].strftime("%Y-%m-%d %a")}: {row["sales"]:>5} units{flag}')

In [ ]:
# ── 7-Day Forecast Bar Chart ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

colors = ['#667eea' if i == 0 else ('#f87171' if row['holiday'] else '#a5b4fc')
          for i, (_, row) in enumerate(forecast_df.iterrows())]

bars = ax.bar(forecast_df['day'], forecast_df['sales'],
              color=colors, width=0.6, edgecolor='white', linewidth=1.5)

# Value labels on bars
for bar, val in zip(bars, forecast_df['sales']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5,
            str(val), ha='center', va='bottom', fontsize=10, fontweight='bold', color='#444')

ax.set_ylabel('Predicted Sales (units)', fontsize=11)
ax.set_title(
    f'7-Day Sales Forecast  ·  Store {FORECAST_STORE}  ·  Product {FORECAST_PRODUCT}\n'
    f'Starting {FORECAST_START}  |  Promotion: {"Yes" if PROMO_ACTIVE else "No"}',
    fontsize=13, fontweight='bold'
)
ax.set_ylim(0, max(forecast_df['sales']) * 1.2)
ax.spines[['top', 'right']].set_visible(False)
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#667eea', label='Forecast start'),
    Patch(facecolor='#a5b4fc', label='Regular day'),
    Patch(facecolor='#f87171', label='Holiday'),
]
ax.legend(handles=legend_elements, loc='upper right', framealpha=0.8)

fig.patch.set_facecolor('#fafbff')
ax.set_facecolor('#fafbff')
plt.tight_layout()
plt.show()

---
## 💾 Save & Load Model (Optional)

In [ ]:
# ── Save model to disk ────────────────────────────────────────────────────────
os.makedirs('models', exist_ok=True)

payload = {'model': model, 'feature_cols': FEATURE_COLS}
joblib.dump(payload, 'models/model.pkl')
print('✅ Model saved to models/model.pkl')

# ── Reload and verify ─────────────────────────────────────────────────────────
loaded   = joblib.load('models/model.pkl')
m_loaded = loaded['model']
f_loaded = loaded['feature_cols']

test_pred = m_loaded.predict(X_test[:5])
print(f'\nVerification — first 5 test predictions from reloaded model:')
print(f'  {[round(p) for p in test_pred]}')

---
## 🎯 Try Your Own Prediction

Edit the values below and run the cell to get a prediction!

In [ ]:
# ── Edit these values ─────────────────────────────────────────────────────────
MY_STORE     = 2          # Options: 1, 2, 3
MY_PRODUCT   = 103        # Options: 101, 102, 103, 104
MY_DATE      = '2025-08-10'  # Format: YYYY-MM-DD
MY_PROMOTION = 1          # 1 = Yes, 0 = No
MY_HOLIDAY   = 0          # 1 = Yes, 0 = No

# Optional: provide historical sales for better accuracy
MY_LAG_1  = None   # sales yesterday  (or set a number, e.g. 210)
MY_LAG_7  = None   # sales 7 days ago
MY_LAG_14 = None   # sales 14 days ago
# ─────────────────────────────────────────────────────────────────────────────

result = predict_sales(
    store_id=MY_STORE,
    product_id=MY_PRODUCT,
    date=MY_DATE,
    promotion=MY_PROMOTION,
    holiday=MY_HOLIDAY,
    sales_lag_1=MY_LAG_1,
    sales_lag_7=MY_LAG_7,
    sales_lag_14=MY_LAG_14
)

day_name = pd.to_datetime(MY_DATE).strftime('%A')

print('┌' + '─' * 45 + '┐')
print(f'│  Retail Sales Forecast Result            │')
print('├' + '─' * 45 + '┤')
print(f'│  Store        : {MY_STORE:<28}│')
print(f'│  Product      : {MY_PRODUCT:<28}│')
print(f'│  Date         : {MY_DATE} ({day_name:<11})│')
print(f'│  Promotion    : {"Yes" if MY_PROMOTION else "No":<28}│')
print(f'│  Holiday      : {"Yes" if MY_HOLIDAY else "No":<28}│')
print('├' + '─' * 45 + '┤')
print(f'│  Predicted Sales : {result:>5} units             │')
print('└' + '─' * 45 + '┘')

---
## 📋 Summary

| Metric | Value |
|--------|-------|
| Algorithm | Random Forest Regressor |
| Training samples | ~10,500 |
| Test samples | ~2,600 |
| Features | 17 |
| MAE | ~12–15 units |
| RMSE | ~18–22 units |
| R² Score | ~0.92 |

### Key Insights
- **Lag features** (especially `sales_lag_1`) are the most important predictors
- **Rolling averages** help the model understand trend and seasonality
- **Promotions** add ~50 units to expected sales
- **Weekends** consistently outperform weekdays by ~30 units

---
*Built with Python · Scikit-learn · Pandas · Matplotlib*  
*GitHub: [G0kulC/retail-sales-forecasting](https://github.com/G0kulC/retail-sales-forecasting)*